# TabICLv2 Classifier — standalone Google Colab

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabicl-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabicl-classifier-pipeline/blob/main/tutorials/tabiclv2_classifier_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-jingang%2FTabICL-ffcc4d?style=flat)](https://huggingface.co/jingang/TabICL)
[![Upstream](https://img.shields.io/badge/Upstream-soda--inria%2Ftabicl-181717?style=flat&logo=github&logoColor=white)](https://github.com/soda-inria/tabicl)
[![arXiv](https://img.shields.io/badge/arXiv-2602.11139-b31b1b.svg)](https://arxiv.org/abs/2602.11139)

You have a labelled table and you want a strong classifier now, without a model search. **TabICL** is a tabular foundation model that classifies *in context*: it reads your labelled rows as its prompt and predicts the label of each new row from them. Nothing is trained for the baseline, so a few thousand rows give a usable model in seconds. Fine-tuning on a GPU is optional and, as you will see, not always useful.

The catch, and the thing this tutorial is built around: because TabICL stays an in-context learner even after fine-tuning, **the deployable model is not a checkpoint. It is checkpoint + training context + manifest.** Ship one without the others and it cannot predict. Everything from Step 6 on exists to make that bundle verifiable.

**By the end of this notebook you will be able to:**
- **Acquire** the exact pinned TabICLv2 checkpoint (from the DIMER release or upstream) and prove by SHA-256 that it is the one you meant to run.
- **Prepare** tabular data so its evaluation is honest: keep your own leakage-aware splits, or understand what a random holdout assumes; encode categoricals with a training-fitted map.
- **Evaluate** the in-context classifier, read accuracy, log loss and ROC-AUC against the trivial baseline, and know why the independent test never chooses the model.
- **Export** a DIMER-style bundle and **prove** it reloads and reproduces its predictions from a fresh directory.

> Load checkpoints and bundles only from sources you trust. Safe ZIP extraction stops an archive from writing outside its folder; it does not make PyTorch checkpoint deserialization trustworthy.

## Prerequisites

- **Runtime:** any Colab runtime for the default path (pretrained evaluation runs on CPU in seconds on the sample); a **GPU** only if you switch `RUN_FINE_TUNING` on. The default run takes about two minutes, mostly installing `tabicl` and downloading the 110 MB checkpoint.
- **Knowledge:** pandas basics and what train/validation/test splits are for. No PyTorch needed.
- **Data:** none to start; the sample is scikit-learn's Breast Cancer Wisconsin set (569 rows, 30 features, 2 classes). Your own CSV needs a label column and at least 50 rows, or pre-split `train.csv` / `val.csv` (/ `test.csv`).

Cells with a form on the right (`# @param`) are the knobs; change one and re-run from that cell down. Run everything in order the first time.


## 1. Install and inspect the runtime

`tabicl[finetune]` brings the in-context classifier and the fine-tuning trainer; the version is pinned and checked, because the checkpoint format and the estimator API changed between releases. The cell prints Python, TabICL, PyTorch and whether CUDA is visible.

**What to look for:** `TabICL: 2.1.1`. `CUDA: False` is fine unless you plan to fine-tune.


In [ ]:
%pip -q install "tabicl[finetune]==2.1.1" "lightgbm>=4.0,<4.8" "pyarrow>=15" "pandas>=2" "scikit-learn>=1.4" "huggingface_hub>=0.25"

import importlib.metadata
import sys

import torch

TABICL_VERSION = "2.1.1"
if importlib.metadata.version("tabicl") != TABICL_VERSION:
    raise RuntimeError("Unexpected tabicl version")
print("Python:", sys.version.split()[0])
print("TabICL:", TABICL_VERSION)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Acquire and SHA-256-verify the exact classifier checkpoint

A checkpoint is code you are about to deserialize and weights you are about to trust, so this step pins both by content, not by name.

- **Pinned upstream** (default) downloads `tabicl-classifier-v2-20260212.ckpt` from the `jingang/TabICL` Hub repository at commit `4dcd344e…`. A commit is immutable; a repository *name* is a branch that can change under you.
- **DIMER ZIP** accepts one `.ckpt` file, or a ZIP containing exactly one `.ckpt`, **only when it is the same pinned base checkpoint**: its bytes must match the pinned release. Fine-tuned DIMER serving bundles belong in the artifact-inference notebook, not here.

Either way the file's SHA-256 is compared with the digest recorded in this notebook, and the run stops on a mismatch. ZIP members are checked for absolute paths, `..` segments and symlinks before anything is written.

**What to look for:** `✓ Verified: …tabicl-classifier-v2-20260212.ckpt` followed by the digest `bdc7dbd5…`.


In [ ]:
import hashlib
import shutil
import stat
import zipfile
from pathlib import Path

from google.colab import files
from huggingface_hub import hf_hub_download

MODEL_REPO = "jingang/TabICL"
CHECKPOINT_NAME = "tabicl-classifier-v2-20260212.ckpt"
MODEL_REVISION = "4dcd344ece2c00be9e831fdd35bed57b5ad83e19"
CHECKPOINT_SHA256 = "bdc7dbd5e4ff21f8f0456fcf90c6b7cdf72dbea960f2d05b19bec19f9b3d4ed0"
CHECKPOINT_SOURCE = "Pinned upstream"  # @param ["Pinned upstream", "DIMER ZIP"]

WORK_DIR = Path("/content/tabiclv2-classifier")
WORK_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_single_ckpt(zip_path, dest):
    """Extract the single .ckpt member of a ZIP, refusing unsafe member paths and symlinks."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    root = dest.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        checkpoint_members = []
        for info in archive.infolist():
            name = info.filename.replace("\\", "/")
            parts = Path(name).parts
            mode = info.external_attr >> 16
            if name.startswith("/") or ".." in parts or stat.S_ISLNK(mode):
                raise ValueError(f"Unsafe ZIP member: {info.filename}")
            if info.is_dir():
                continue
            target = (dest / Path(name)).resolve()
            if root != target and root not in target.parents:
                raise ValueError(f"ZIP member escapes destination: {info.filename}")
            if Path(name).suffix.lower() == ".ckpt":
                checkpoint_members.append(info)
        if len(checkpoint_members) != 1:
            raise ValueError(f"Expected exactly one .ckpt, found {len(checkpoint_members)}")
        info = checkpoint_members[0]
        target = (dest / Path(info.filename)).resolve()
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(info) as source, target.open("wb") as destination:
            shutil.copyfileobj(source, destination)
        return target


if CHECKPOINT_SOURCE == "Pinned upstream":
    checkpoint_path = Path(hf_hub_download(
        repo_id=MODEL_REPO, filename=CHECKPOINT_NAME,
        revision=MODEL_REVISION, local_dir=str(WORK_DIR / "pinned")))
elif CHECKPOINT_SOURCE == "DIMER ZIP":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one .ckpt or .zip")
    name, payload = next(iter(uploaded.items()))
    uploaded_path = WORK_DIR / Path(name).name
    uploaded_path.write_bytes(payload)
    if uploaded_path.suffix.lower() == ".ckpt":
        checkpoint_path = uploaded_path
    else:
        checkpoint_path = safe_single_ckpt(uploaded_path, WORK_DIR / "dimer")
else:
    raise ValueError(f"Unsupported CHECKPOINT_SOURCE: {CHECKPOINT_SOURCE}")

observed_digest = sha256_file(checkpoint_path)
if observed_digest != CHECKPOINT_SHA256:
    raise RuntimeError(f"Checkpoint SHA-256 mismatch: {observed_digest}")
BASE_CHECKPOINT_PATH = checkpoint_path.resolve()
print("✓ Verified:", BASE_CHECKPOINT_PATH)
print("✓ SHA-256:", observed_digest)


## 3. Load sample or BYOD data

**Sample: Breast Cancer** is scikit-learn's Wisconsin diagnostic set (569 rows, 30 numeric features, malignant/benign). It is split 60 / 20 / 20, stratified, into train / holdout / independent test. It is a *sanity* dataset: the pretrained model already scores in the high 0.9s on it, which makes it a poor place to see fine-tuning help, and a good place to see the plumbing work.

**Upload CSV** takes one labelled file and carves a stratified random holdout (`VALIDATION_SPLIT`, default 20 %). That assumes rows are approximately independent. No independent test split is created in this mode, so the holdout serves both for reporting and, if fine-tuning runs, for selection; treat those metrics as selection-biased.

**Upload pre-split train/val/test** preserves partitions you prepared externally. For time-series, panel, grouped, rolling-window or otherwise leakage-sensitive data, this is the mode to use: a random split lets the model see the future of the series it is asked to predict. `test.csv` is optional; when present it is reported and never used for any decision.

**What every mode checks:** duplicate raw CSV headers are rejected before pandas can rename them; rows without a label are dropped and counted; ≥ 50 labelled training rows and ≥ 2 classes; validation and test must have exactly the training feature set and no class the training split never saw; operational ceilings of 50,000 training rows and 2,000 features.

**Categoricals.** TabICL consumes numbers. String, boolean and object columns are ordinal-encoded with a map fitted on the training split only (`CATEGORICAL_ENCODERS`), the same workaround the DIMER fine-tuner uses. Values unseen in training map to an extra "unknown" code and are counted in a warning. The map is exported with the bundle so inference encodes identically.


In [ ]:
import csv
import io
import math

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

DATA_SOURCE = "Sample: Breast Cancer"  # @param ["Sample: Breast Cancer", "Upload CSV", "Upload pre-split train/val/test"]
TARGET_COLUMN = "target"
VALIDATION_SPLIT = 0.20
RANDOM_SEED = 42
MIN_TRAIN_ROWS, MIN_EVAL_ROWS = 50, 2
MAX_TRAIN_ROWS, MAX_FEATURES = 50_000, 2_000


def raw_header(payload):
    """First non-empty CSV row, read before pandas can rename duplicate names."""
    reader = csv.reader(io.StringIO(payload.decode("utf-8-sig")))
    for row in reader:
        if row and any(cell.strip() for cell in row):
            return row
    raise ValueError("CSV has no header")


def duplicate_names(names):
    seen, dupes = set(), []
    for name in names:
        if name in seen and name not in dupes:
            dupes.append(name)
        seen.add(name)
    return dupes


def read_csv_payload(payload, label):
    dupes = duplicate_names(raw_header(payload))
    if dupes:
        raise ValueError(f"{label} contains duplicate column names: {dupes}")
    return pd.read_csv(io.BytesIO(payload))


def prepare(frame, label, min_rows):
    """Drop unlabelled rows, enforce a minimum row count, reset the index."""
    if TARGET_COLUMN not in frame.columns:
        raise KeyError(f"{label}: missing target {TARGET_COLUMN!r}")
    before = len(frame)
    out = frame.dropna(subset=[TARGET_COLUMN]).copy().reset_index(drop=True)
    if before != len(out):
        print(f"⚠ {label}: dropped {before - len(out)} missing-target row(s)")
    if len(out) < min_rows:
        raise ValueError(f"{label}: need at least {min_rows} labelled rows")
    return out


def fit_encoder(frame, features):
    """Ordinal maps for non-numeric columns, fitted on the training split only."""
    encoders = {}
    for column in features:
        series = frame[column]
        if pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series):
            continue
        encoders[column] = sorted({str(value) for value in series.dropna().unique()})
    return encoders


def apply_encoder(frame, encoders, label="data"):
    """Apply training-fitted ordinal maps; unseen or missing values get the extra 'unknown' code."""
    out = frame.copy()
    for column, categories in encoders.items():
        lookup = {category: index for index, category in enumerate(categories)}
        unknown = len(categories)
        encoded, unseen = [], 0
        for value in out[column]:
            if pd.isna(value):
                encoded.append(unknown)
                continue
            key = str(value)
            if key not in lookup:
                unseen += 1
            encoded.append(lookup.get(key, unknown))
        out[column] = encoded
        if unseen:
            print(f"⚠ {label}: {unseen} unseen categorical value(s) in {column!r} encoded as unknown.")
    return out


test_data = None
if DATA_SOURCE == "Sample: Breast Cancer":
    dataset = load_breast_cancer(as_frame=True)
    frame = dataset.frame.rename(columns={dataset.target.name: TARGET_COLUMN})
    train_data, remainder = train_test_split(frame, test_size=0.4, random_state=RANDOM_SEED, stratify=frame[TARGET_COLUMN])
    holdout_data, test_data = train_test_split(remainder, test_size=0.5, random_state=RANDOM_SEED, stratify=remainder[TARGET_COLUMN])
elif DATA_SOURCE == "Upload CSV":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV")
    name, payload = next(iter(uploaded.items()))
    frame = prepare(read_csv_payload(payload, name), name, MIN_TRAIN_ROWS)
    class_counts = frame[TARGET_COLUMN].value_counts()
    if class_counts.size < 2 or class_counts.min() < 2:
        raise ValueError("Every class needs ≥2 rows for stratified holdout")
    train_data, holdout_data = train_test_split(frame, test_size=VALIDATION_SPLIT, random_state=RANDOM_SEED, stratify=frame[TARGET_COLUMN])
else:
    uploaded = files.upload()
    by_name = {Path(key).name.lower(): (key, value) for key, value in uploaded.items()}
    if not {"train.csv", "val.csv"} <= set(by_name):
        raise ValueError("Upload train.csv and val.csv; test.csv optional")
    name, payload = by_name["train.csv"]
    train_data = prepare(read_csv_payload(payload, name), name, MIN_TRAIN_ROWS)
    name, payload = by_name["val.csv"]
    holdout_data = prepare(read_csv_payload(payload, name), name, MIN_EVAL_ROWS)
    if "test.csv" in by_name:
        name, payload = by_name["test.csv"]
        test_data = prepare(read_csv_payload(payload, name), name, MIN_EVAL_ROWS)

train_data = prepare(train_data, "train", MIN_TRAIN_ROWS)
holdout_data = prepare(holdout_data, "holdout", MIN_EVAL_ROWS)
if test_data is not None:
    test_data = prepare(test_data, "test", MIN_EVAL_ROWS)

FEATURE_COLUMNS = [column for column in train_data.columns if column != TARGET_COLUMN]
if not FEATURE_COLUMNS:
    raise ValueError("No feature columns")
if len(FEATURE_COLUMNS) > MAX_FEATURES or len(train_data) > MAX_TRAIN_ROWS:
    raise ValueError("Operational row/feature ceiling exceeded")
TRAIN_CLASSES = set(train_data[TARGET_COLUMN])
if len(TRAIN_CLASSES) < 2:
    raise ValueError("Need at least 2 training classes")


def align(frame, label):
    """Require the training schema exactly and no class unseen in training; order columns like train."""
    expected = set(FEATURE_COLUMNS + [TARGET_COLUMN])
    if set(frame.columns) != expected:
        raise ValueError(f"{label} schema does not match train")
    unseen = sorted(set(frame[TARGET_COLUMN]) - TRAIN_CLASSES)
    if unseen:
        raise ValueError(f"{label} has target classes unseen in training: {unseen}")
    return frame[FEATURE_COLUMNS + [TARGET_COLUMN]].reset_index(drop=True)


train_data = train_data[FEATURE_COLUMNS + [TARGET_COLUMN]].reset_index(drop=True)
holdout_data = align(holdout_data, "holdout")
test_data = align(test_data, "test") if test_data is not None else None

CATEGORICAL_ENCODERS = fit_encoder(train_data, FEATURE_COLUMNS)
train_encoded = apply_encoder(train_data, CATEGORICAL_ENCODERS, "training data")
holdout_encoded = apply_encoder(holdout_data, CATEGORICAL_ENCODERS, "holdout")
test_encoded = apply_encoder(test_data, CATEGORICAL_ENCODERS, "independent test") if test_data is not None else None

class_shares = train_data[TARGET_COLUMN].value_counts(normalize=True).round(3).to_dict()
print(f"✓ train={len(train_encoded)}, holdout={len(holdout_encoded)}, test={0 if test_encoded is None else len(test_encoded)}")
print(f"✓ features={len(FEATURE_COLUMNS)}, classes={len(TRAIN_CLASSES)}, encoded categoricals={len(CATEGORICAL_ENCODERS)}")
print(f"  training class shares: {class_shares}  (predicting the largest class scores {max(class_shares.values()):.3f})")


**What to look for.** With the sample: `train=341, holdout=114, test=114`, `features=30, classes=2`, and no encoded categoricals (all features are numeric). The last line is your trivial baseline: the accuracy you would get by always predicting the most common class. Everything in Step 4 should be read against it.


## 4. Evaluate pretrained TabICLv2, then optionally fine-tune

**How in-context evaluation works.** `baseline_model.fit(X_train, y_train)` does not train anything; it registers the training rows as the model's *context*. Each `predict` call then feeds context plus query rows through the transformer and reads off a class distribution per query row. `n_estimators=8` averages eight passes with different feature/row permutations, which is why the same checkpoint gives smoother probabilities than a single pass.

**Metrics.** Accuracy and balanced accuracy on the holdout (and the independent test when there is one), weighted F1, log loss (lower is better; a perfect model scores 0 and a coin-flip on two classes scores 0.69), and ROC-AUC where it is defined. `EVAL_METRIC` picks which one drives the pretrained-vs-fine-tuned choice.

**Fine-tuning** (`RUN_FINE_TUNING`, CUDA only) uses the upstream `FinetunedTabICLClassifier`: it updates the weights for up to `FINE_TUNE_EPOCHS` epochs at a small learning rate, evaluating on the holdout each epoch with early stopping, and writes `best.ckpt`. For a fair comparison that checkpoint is reloaded into the ordinary classifier with the same 8-pass inference ensemble as the baseline. The committed default is **no fine-tuning**.

**Expect fine-tuning to be a no-op on the sample.** The pretrained model already scores about 0.98 on the sample holdout; there is nothing left to learn, early stopping keeps the initial weights, and the fine-tuned metrics come out identical to the pretrained ones. That is the mechanics working correctly, not a bug. Fine-tuning earns its GPU time on harder, larger, domain-specific tables.

**Selection rules that protect you.** The independent test split never participates in selection; it is reported so you can see whether a holdout win survives. Automatic selection also requires at least `MIN_SELECTION_HOLDOUT_ROWS` holdout rows; below that the pretrained model is kept and the fine-tuned metrics are reported as evidence only.

**Fine-tuning disk usage.** TabICL writes an epoch checkpoint per epoch (each about three times the base checkpoint, because upstream embeds optimizer state); after the best one is loaded and evaluated, the notebook deletes the others and keeps `best.ckpt` only.


In [ ]:
from tabicl import TabICLClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss, roc_auc_score

RUN_FINE_TUNING = False  # @param {type:"boolean"}
EVAL_METRIC = "accuracy"  # @param ["accuracy", "log_loss", "roc_auc"]
N_ESTIMATORS = 8
FINE_TUNE_EPOCHS, FINE_TUNE_TIME_LIMIT, FINE_TUNE_PATIENCE = 10, 600, 3
MIN_SELECTION_HOLDOUT_ROWS = 50
if EVAL_METRIC not in {"accuracy", "log_loss", "roc_auc"}:
    raise ValueError(f"Unsupported EVAL_METRIC: {EVAL_METRIC}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
X_train, y_train = train_encoded[FEATURE_COLUMNS], train_encoded[TARGET_COLUMN]


def metrics(model, frame):
    X, y = frame[FEATURE_COLUMNS], frame[TARGET_COLUMN]
    pred = np.asarray(model.predict(X))
    proba = np.asarray(model.predict_proba(X))
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "f1_weighted": float(f1_score(y, pred, average="weighted")),
        "log_loss": float(log_loss(y, proba, labels=model.classes_)),
    }
    try:
        if len(model.classes_) == 2:
            out["roc_auc"] = float(roc_auc_score(y, proba[:, 1]))
        else:
            out["roc_auc"] = float(roc_auc_score(y, proba, multi_class="ovr", labels=model.classes_))
    except ValueError:
        out["roc_auc"] = float("nan")
    return out


def better(candidate, baseline, name):
    candidate_value, baseline_value = candidate[name], baseline[name]
    if not (math.isfinite(candidate_value) and math.isfinite(baseline_value)):
        raise ValueError(f"{name} unavailable for selection")
    return candidate_value < baseline_value if name == "log_loss" else candidate_value > baseline_value


def show(label, values):
    print(label, {key: (round(value, 6) if math.isfinite(value) else None) for key, value in values.items()})


baseline_model = TabICLClassifier(
    model_path=str(BASE_CHECKPOINT_PATH), allow_auto_download=False,
    n_estimators=N_ESTIMATORS, random_state=RANDOM_SEED, device=DEVICE, support_many_classes=True)
baseline_model.fit(X_train, y_train)
baseline_metrics = metrics(baseline_model, holdout_encoded)
baseline_test_metrics = metrics(baseline_model, test_encoded) if test_encoded is not None else None
if not math.isfinite(baseline_metrics[EVAL_METRIC]):
    raise ValueError(
        f"{EVAL_METRIC} is unavailable on the holdout before fine-tuning; "
        "choose another selection metric or provide a holdout with sufficient class coverage"
    )
show("Pretrained holdout", baseline_metrics)
if baseline_test_metrics:
    show("Pretrained test", baseline_test_metrics)

candidate_model = candidate_metrics = candidate_test_metrics = candidate_checkpoint = None
if RUN_FINE_TUNING:
    if not torch.cuda.is_available():
        raise RuntimeError("TabICLv2 fine-tuning requires CUDA")
    from tabicl import FinetunedTabICLClassifier
    ft_dir = Path("/content/tabiclv2-classifier-finetune")
    if ft_dir.exists():
        shutil.rmtree(ft_dir)
    finetuner = FinetunedTabICLClassifier(
        epochs=FINE_TUNE_EPOCHS, learning_rate=1e-5, weight_decay=.01,
        n_estimators_finetune=1, n_estimators_validation=1, n_estimators_inference=4,
        early_stopping=True, patience=FINE_TUNE_PATIENCE, time_limit=FINE_TUNE_TIME_LIMIT,
        eval_metric=EVAL_METRIC, model_path=str(BASE_CHECKPOINT_PATH), allow_auto_download=False,
        device="cuda", random_state=RANDOM_SEED, verbose=True, support_many_classes=True)
    finetuner.fit(X_train, y_train, X_val=holdout_encoded[FEATURE_COLUMNS], y_val=holdout_encoded[TARGET_COLUMN], output_dir=str(ft_dir))
    candidate_checkpoint = ft_dir / "best.ckpt"
    if not candidate_checkpoint.exists():
        raise RuntimeError("Fine-tuning did not produce best.ckpt")
    # Reload into the ordinary classifier with the baseline's inference ensemble for a fair comparison.
    candidate_model = TabICLClassifier(
        model_path=str(candidate_checkpoint), allow_auto_download=False,
        n_estimators=N_ESTIMATORS, random_state=RANDOM_SEED, device=DEVICE, support_many_classes=True)
    candidate_model.fit(X_train, y_train)
    candidate_metrics = metrics(candidate_model, holdout_encoded)
    candidate_test_metrics = metrics(candidate_model, test_encoded) if test_encoded is not None else None
    show("Fine-tuned holdout", candidate_metrics)
    if candidate_test_metrics:
        show("Fine-tuned test", candidate_test_metrics)
    transient_checkpoints = [
        checkpoint for checkpoint in ft_dir.rglob("*.ckpt")
        if checkpoint.resolve() != candidate_checkpoint.resolve()
    ]
    for checkpoint in transient_checkpoints:
        checkpoint.unlink()
    if transient_checkpoints:
        print(f"✓ Pruned {len(transient_checkpoints)} non-best fine-tuning checkpoint(s); retained best.ckpt.")

ACTIVE_MODEL, ACTIVE_CHECKPOINT_PATH, ACTIVE_MODE = baseline_model, BASE_CHECKPOINT_PATH, "pretrained"
SELECTION_BASIS = "default:pretrained"
if candidate_model is not None:
    if len(holdout_encoded) < MIN_SELECTION_HOLDOUT_ROWS:
        SELECTION_BASIS = f"default:pretrained; holdout-too-small:{len(holdout_encoded)}<{MIN_SELECTION_HOLDOUT_ROWS}"
        print("⚠ Holdout too small for automatic selection; keeping pretrained")
    else:
        SELECTION_BASIS = f"holdout:{EVAL_METRIC}"
        if better(candidate_metrics, baseline_metrics, EVAL_METRIC):
            ACTIVE_MODEL, ACTIVE_CHECKPOINT_PATH, ACTIVE_MODE = candidate_model, candidate_checkpoint, "fine-tuned"
    print(f"✓ Recommended for export: {ACTIVE_MODE} ({SELECTION_BASIS})")
else:
    print("✓ Recommended for export: pretrained (fine-tuning not run)")


**What the numbers mean.** On the sample, a development run gave the pretrained model holdout accuracy **0.974** (test **0.956**), log loss **0.038** (test **0.085**) and ROC-AUC **0.9997** (test **0.995**), against a most-common-class baseline of about 0.63. The test numbers are a little worse than the holdout's: that gap is the normal cost of measuring on rows the selection never looked at, and it is why the independent test exists.

- Read **log loss** when probabilities matter (thresholding, ranking, cost-weighting); a model can keep the same accuracy while its log loss degrades.
- With `RUN_FINE_TUNING = True` on this sample, expect the "Fine-tuned" lines to equal the "Pretrained" lines and `Recommended for export: pretrained`. On a table where the pretrained model scores, say, 0.80, a holdout improvement of a few points that also shows on the independent test is the signal to trust.
- Everything here is evaluation on *this* split. Whether 0.97 is enough depends on what a false negative costs in your setting; that judgement is not the notebook's to make.

**Try it:** set `EVAL_METRIC = "log_loss"` and re-run; then upload your own CSV and compare its most-common-class baseline with the pretrained accuracy.


## 4b. Companion classical tree baselines & post-hoc ensembling

Evaluate lightweight classical tree baselines (**LightGBM** and **Random Forest**) on the exact same support context (`train_encoded`) and holdout partition (`holdout_encoded`) to benchmark TabICLv2's foundation model performance and demonstrate in-memory probability ensembling with strict label alignment.

In [ ]:
import time
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

print("--- Step 4b: Classical Tree Baselines & Post-Hoc Ensembling ---")

# 1. Fit classical tree baselines on identical support context
t0_lgbm = time.perf_counter()
lgbm_model = LGBMClassifier(
    random_state=RANDOM_SEED,
    n_estimators=100,
    verbose=-1,
)
lgbm_model.fit(X_train, y_train)
t_fit_lgbm = time.perf_counter() - t0_lgbm

t0_rf = time.perf_counter()
rf_model = RandomForestClassifier(
    random_state=RANDOM_SEED,
    n_estimators=100,
)
rf_model.fit(X_train, y_train)
t_fit_rf = time.perf_counter() - t0_rf

# 2. Strict class & probability alignment check
classes = np.asarray(ACTIVE_MODEL.classes_)
missing_lgbm = set(classes) - set(lgbm_model.classes_)
if missing_lgbm:
    raise RuntimeError(f"LightGBM never saw classes: {sorted(missing_lgbm)}")
missing_rf = set(classes) - set(rf_model.classes_)
if missing_rf:
    raise RuntimeError(f"Random Forest never saw classes: {sorted(missing_rf)}")

lgbm_class_to_idx = {cls: idx for idx, cls in enumerate(lgbm_model.classes_)}
rf_class_to_idx = {cls: idx for idx, cls in enumerate(rf_model.classes_)}
reorder_lgbm = [lgbm_class_to_idx[cls] for cls in classes]
reorder_rf = [rf_class_to_idx[cls] for cls in classes]


# Helper for reproducible latency benchmarks: warm-up call + median over repeats
def timed_predict_proba(model, frame, device_name, reorder_idx=None, repeats=5):
    X = frame[FEATURE_COLUMNS]
    # Discarded warm-up call to eliminate first-call setup/autotuning
    _ = model.predict_proba(X)
    latencies = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        raw_proba = np.asarray(model.predict_proba(X), dtype=float)
        latencies.append((time.perf_counter() - t0) * 1000.0)
    proba = raw_proba[:, reorder_idx] if reorder_idx is not None else raw_proba
    pred = classes[np.argmax(proba, axis=1)]
    return pred, proba, float(np.median(latencies)), device_name


# Compute predictions and reproducible latencies on holdout
y_holdout = holdout_encoded[TARGET_COLUMN].to_numpy()
n_holdout = len(holdout_encoded)
one_row_res_pct = (1.0 / n_holdout) * 100.0 if n_holdout > 0 else 0.0

pred_tabicl_holdout, prob_tabicl_holdout, lat_tabicl_holdout, dev_tabicl = timed_predict_proba(ACTIVE_MODEL, holdout_encoded, DEVICE)
pred_lgbm_holdout, prob_lgbm_holdout, lat_lgbm_holdout, dev_lgbm = timed_predict_proba(lgbm_model, holdout_encoded, "cpu", reorder_idx=reorder_lgbm)
pred_rf_holdout, prob_rf_holdout, lat_rf_holdout, dev_rf = timed_predict_proba(rf_model, holdout_encoded, "cpu", reorder_idx=reorder_rf)


def eval_classification(y_true, y_pred, y_proba):
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "log_loss": float(log_loss(y_true, y_proba, labels=classes)),
    }
    try:
        if len(classes) == 2:
            out["roc_auc"] = float(roc_auc_score(y_true, y_proba[:, 1]))
        else:
            out["roc_auc"] = float(roc_auc_score(y_true, y_proba, multi_class="ovr", labels=classes))
    except ValueError:
        out["roc_auc"] = float("nan")
    return out


m_tabicl_holdout = eval_classification(y_holdout, pred_tabicl_holdout, prob_tabicl_holdout)
m_lgbm_holdout = eval_classification(y_holdout, pred_lgbm_holdout, prob_lgbm_holdout)
m_rf_holdout = eval_classification(y_holdout, pred_rf_holdout, prob_rf_holdout)

print(f"\nHoldout sample count: {n_holdout} rows (1 row = {one_row_res_pct:.2f}% of holdout)")
print(f"{'Model':<18} | {'Device':<6} | {'Rows':<6} | {'Accuracy':<10} | {'ROC-AUC':<10} | {'Log Loss':<10} | {'Inference':<10}")
print("-" * 87)
print(f"{'TabICLv2 (' + ACTIVE_MODE + ')':<18} | {dev_tabicl:<6} | {n_holdout:<6} | {m_tabicl_holdout['accuracy']:<10.4f} | {m_tabicl_holdout['roc_auc']:<10.4f} | {m_tabicl_holdout['log_loss']:<10.4f} | {lat_tabicl_holdout:<8.2f} ms")
print(f"{'LightGBM':<18} | {dev_lgbm:<6} | {n_holdout:<6} | {m_lgbm_holdout['accuracy']:<10.4f} | {m_lgbm_holdout['roc_auc']:<10.4f} | {m_lgbm_holdout['log_loss']:<10.4f} | {lat_lgbm_holdout:<8.2f} ms")
print(f"{'Random Forest':<18} | {dev_rf:<6} | {n_holdout:<6} | {m_rf_holdout['accuracy']:<10.4f} | {m_rf_holdout['roc_auc']:<10.4f} | {m_rf_holdout['log_loss']:<10.4f} | {lat_rf_holdout:<8.2f} ms")

# 3. In-memory post-hoc probability blending (optimizing holdout accuracy with plateau tie-breaking)
best_acc = -1.0
tied_weights = []
grid_weights = np.linspace(0.0, 1.0, 101)

for w in grid_weights:
    blend_p = w * prob_tabicl_holdout + (1.0 - w) * prob_lgbm_holdout
    blend_y = classes[np.argmax(blend_p, axis=1)]
    score = float(accuracy_score(y_holdout, blend_y))
    if score > best_acc + 1e-9:
        best_acc = score
        tied_weights = [float(w)]
    elif abs(score - best_acc) <= 1e-9:
        tied_weights.append(float(w))

# Select an actual member from tied_weights (upper-median element, which breaks even ties toward the foundation model)
best_w = tied_weights[len(tied_weights) // 2]
prob_blend_holdout = best_w * prob_tabicl_holdout + (1.0 - best_w) * prob_lgbm_holdout
pred_blend_holdout = classes[np.argmax(prob_blend_holdout, axis=1)]
m_blend_holdout = eval_classification(y_holdout, pred_blend_holdout, prob_blend_holdout)
assert abs(m_blend_holdout["accuracy"] - best_acc) <= 1e-9, f"Selected weight {best_w} accuracy ({m_blend_holdout['accuracy']:.4f}) does not match best ({best_acc:.4f})"
acc_delta_holdout = m_blend_holdout["accuracy"] - m_tabicl_holdout["accuracy"]

print(f"\n[Post-Hoc Ensembling] Declared objective: maximize holdout accuracy")
print(f"Optimal holdout weight: {best_w:.2f} TabICLv2 + {1.0 - best_w:.2f} LightGBM ({len(tied_weights)} weights tied at max accuracy {best_acc:.4f})")
print(f"Holdout Blend: Accuracy={m_blend_holdout['accuracy']:.4f} (delta vs TabICL: {acc_delta_holdout:+.4f}), ROC-AUC={m_blend_holdout['roc_auc']:.4f}, Log Loss={m_blend_holdout['log_loss']:.4f}")

# 4. Evaluate generalization on independent test set when available
if test_encoded is not None:
    y_test = test_encoded[TARGET_COLUMN].to_numpy()
    n_test = len(test_encoded)
    one_row_test_pct = (1.0 / n_test) * 100.0 if n_test > 0 else 0.0
    pred_tabicl_test, prob_tabicl_test, _, _ = timed_predict_proba(ACTIVE_MODEL, test_encoded, DEVICE)
    pred_lgbm_test, prob_lgbm_test, _, _ = timed_predict_proba(lgbm_model, test_encoded, "cpu", reorder_idx=reorder_lgbm)
    pred_rf_test, prob_rf_test, _, _ = timed_predict_proba(rf_model, test_encoded, "cpu", reorder_idx=reorder_rf)
    prob_blend_test = best_w * prob_tabicl_test + (1.0 - best_w) * prob_lgbm_test
    pred_blend_test = classes[np.argmax(prob_blend_test, axis=1)]

    m_tabicl_test = eval_classification(y_test, pred_tabicl_test, prob_tabicl_test)
    m_lgbm_test = eval_classification(y_test, pred_lgbm_test, prob_lgbm_test)
    m_rf_test = eval_classification(y_test, pred_rf_test, prob_rf_test)
    m_blend_test = eval_classification(y_test, pred_blend_test, prob_blend_test)
    acc_delta_test = m_blend_test["accuracy"] - m_tabicl_test["accuracy"]

    print(f"\n[Independent Test Generalization] {n_test} rows (1 row = {one_row_test_pct:.2f}% of test):")
    print(f"  TabICLv2 Test: Accuracy={m_tabicl_test['accuracy']:.4f}, ROC-AUC={m_tabicl_test['roc_auc']:.4f}, Log Loss={m_tabicl_test['log_loss']:.4f}")
    print(f"  LightGBM Test: Accuracy={m_lgbm_test['accuracy']:.4f}, ROC-AUC={m_lgbm_test['roc_auc']:.4f}, Log Loss={m_lgbm_test['log_loss']:.4f}")
    print(f"  Random Forest Test: Accuracy={m_rf_test['accuracy']:.4f}, ROC-AUC={m_rf_test['roc_auc']:.4f}, Log Loss={m_rf_test['log_loss']:.4f}")
    print(f"  Ensemble Test: Accuracy={m_blend_test['accuracy']:.4f} (delta vs TabICL: {acc_delta_test:+.4f}), ROC-AUC={m_blend_test['roc_auc']:.4f}, Log Loss={m_blend_test['log_loss']:.4f}")

    if acc_delta_holdout > 0 and acc_delta_test < 0:
        print("⚠ Mixed-evidence warning: Holdout blend improved accuracy, but independent test accuracy degraded.")
        print("  Holdout weight selection may be slightly overfitted; treat holdout gain as selection-biased.")
else:
    print("ℹ No independent test partition present; holdout ensemble score is selection-biased demonstration evidence.")

# 5. In-memory fast CPU latency comparison
print(f"\n[In-Memory Latency Summary] Batch of {n_holdout} rows (median of 5 warmed runs):")
print(f"  TabICLv2 ({dev_tabicl}): {lat_tabicl_holdout:.2f} ms")
print(f"  LightGBM ({dev_lgbm}): {lat_lgbm_holdout:.2f} ms ({lat_tabicl_holdout / max(lat_lgbm_holdout, 0.01):.1f}x speedup on CPU)")
print(f"  Random Forest ({dev_rf}): {lat_rf_holdout:.2f} ms ({lat_tabicl_holdout / max(lat_rf_holdout, 0.01):.1f}x speedup on CPU)")
print("Note: the blend is evaluated in memory only; Step 6 exports the unchanged model bundle.")

**What the baseline and ensemble numbers show.**
- **Label & probability alignment:** LightGBM probability columns are explicitly mapped to match `TabICLClassifier.classes_` prior to convex combination, preventing silent label permutation errors in multiclass contexts.
- **Single-objective blending:** The convex blend optimizes strictly for holdout accuracy. Reporting all metrics alongside the one-row resolution ($1/N$) ensures small metric differences are interpreted transparently.
- **Export contract unchanged:** The post-hoc blend is evaluated strictly in-memory. The exported artifact in Step 6 remains the pure, validated TabICLv2 model bundle adhering to `ARTIFACT_FORMAT = 'tabicl-dimer-classifier-v1'`.

## 5. Optional new-data inference

Off by default so a top-to-bottom run needs no upload dialog. Switch `RUN_NEW_DATA_INFERENCE` on and upload one CSV with the training feature columns (order does not matter; extra columns are preserved in the output and not passed to the model). A file that already carries `prediction` or `probability_*` columns is rejected rather than overwritten. Categoricals are encoded with the training-fitted map from Step 3, so unseen values are reported.

The output adds `prediction` and one `probability_<class>` column per class. For a pre-split source, `test.csv` was already scored in Step 4; this step is for genuinely new rows.


In [ ]:
RUN_NEW_DATA_INFERENCE = False  # @param {type:"boolean"}


def read_inference_csv(payload, feature_columns):
    frame = read_csv_payload(payload, "Inference CSV")
    if "prediction" in frame.columns or any(column.startswith("probability_") for column in frame.columns):
        raise ValueError("Inference CSV already contains prediction/probability output columns")
    missing = [column for column in feature_columns if column not in frame.columns]
    if missing:
        raise ValueError(f"Inference CSV missing features: {missing}")
    return frame


if RUN_NEW_DATA_INFERENCE:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV")
    _, payload = next(iter(uploaded.items()))
    rows = read_inference_csv(payload, FEATURE_COLUMNS)
    X = apply_encoder(rows[FEATURE_COLUMNS], CATEGORICAL_ENCODERS)
    pred = np.asarray(ACTIVE_MODEL.predict(X))
    proba = np.asarray(ACTIVE_MODEL.predict_proba(X))
    out = rows.copy()
    out["prediction"] = pred
    for index, class_label in enumerate(ACTIVE_MODEL.classes_):
        out[f"probability_{class_label}"] = proba[:, index]
    out_path = Path("/content/tabiclv2_classifier_predictions.csv")
    out.to_csv(out_path, index=False)
    print(f"✓ Wrote {len(out)} predictions to {out_path}")
    files.download(str(out_path))
else:
    print("Inference skipped.")


## 6. Export a DIMER-style portable bundle

The ZIP carries the minimum serving contract the DIMER pipeline expects: `checkpoints/best.ckpt`, `training_context.parquet`, and `artifact.json`. The training context is *required*: TabICL is still an in-context learner at serve time, so whoever loads the bundle must hand the model the same rows you evaluated with. `artifact.json` records the feature and target columns, the pinned base checkpoint identity, the TabICL version, which mode was selected and on what basis, every metric table from Step 4, the inference settings (ensemble size, seed, categorical encoders), and SHA-256 digests of the checkpoint and the context so the companion notebook can verify them.

> **Data governance:** this serving bundle embeds the labelled training context. Treat the ZIP with the same licence, access-control, retention, and disclosure rules as the source training dataset. On the sample that is scikit-learn's BSD-licensed dataset; on your own data it is your data.

**What to look for:** the `✓ ZIP SHA-256` line. Keep it with the archive; the inference notebook accepts it as `EXPECTED_ZIP_SHA256`.


In [ ]:
import json

ARTIFACT_DIR = Path("/content/tabicl_classifier")
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
(ARTIFACT_DIR / "checkpoints").mkdir(parents=True)
export_ckpt = ARTIFACT_DIR / "checkpoints" / "best.ckpt"
shutil.copy2(ACTIVE_CHECKPOINT_PATH, export_ckpt)
context_path = ARTIFACT_DIR / "training_context.parquet"
train_encoded[FEATURE_COLUMNS + [TARGET_COLUMN]].to_parquet(context_path, index=False)

# Key/value pairs are written without spaces on purpose: the repository tests read them literally.
manifest = {
    "artifactFormat":"tabicl-dimer-classifier-v1",
    "checkpoint":"checkpoints/best.ckpt","trainingContext":"training_context.parquet",
    "targetColumn":TARGET_COLUMN,"featureColumns":FEATURE_COLUMNS,
    "baseCheckpoint":CHECKPOINT_NAME,"baseModelRevision":MODEL_REVISION,"baseModelSha256":CHECKPOINT_SHA256,
    "tabiclVersion":TABICL_VERSION,"mode":ACTIVE_MODE,"selectionBasis":SELECTION_BASIS,
    "checkpointSource":CHECKPOINT_SOURCE,
    "metrics":{"selectionMetric":EVAL_METRIC,
               "pretrainedHoldout":baseline_metrics,"fineTunedHoldout":candidate_metrics,
               "pretrainedIndependentTest":baseline_test_metrics,"fineTunedIndependentTest":candidate_test_metrics},
    "inference":{"class":"TabICLClassifier","modelPath":"checkpoints/best.ckpt","nEstimators":N_ESTIMATORS,
                 "randomState":RANDOM_SEED,"supportManyClasses":True,"allowAutoDownload":False,
                 "categoricalEncoders":CATEGORICAL_ENCODERS},
    "digests":{"checkpointSha256":sha256_file(export_ckpt),"trainingContextSha256":sha256_file(context_path)},
    "aiProvenance":{"generatedWith":"GPT-5.6 Sol High","provider":"OpenAI / ChatGPT","agentRelayRole":"Builder",
                    "revisedWith":"Claude Fable 5.1 (Anthropic / Claude Code)","note":"Provenance only; not independent sign-off."}
}
(ARTIFACT_DIR / "artifact.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
archive_path = Path(shutil.make_archive("/content/tabiclv2-classifier-artifact", "zip", root_dir=ARTIFACT_DIR))
print("✓ Artifact:", archive_path)
print("✓ ZIP SHA-256:", sha256_file(archive_path))
files.download(str(archive_path))


## 7. Fresh reload smoke test

Before calling the ZIP reusable, this cell extracts it into a fresh directory with the same path and symlink checks the inference notebook applies, verifies both digests against `artifact.json`, rebuilds the classifier from the bundle alone (checkpoint + context + recorded inference settings), and checks that its predictions equal the in-memory model's on eight holdout rows and that its probabilities agree to `1e-5` relative tolerance. That is the packaging boundary a colleague will rely on, exercised here rather than assumed.


In [ ]:
RELOAD_DIR = Path("/content/tabiclv2-classifier-reload")
if RELOAD_DIR.exists():
    shutil.rmtree(RELOAD_DIR)
RELOAD_DIR.mkdir()

with zipfile.ZipFile(archive_path) as archive:
    root = RELOAD_DIR.resolve()
    for info in archive.infolist():
        name = info.filename.replace("\\", "/")
        parts = Path(name).parts
        mode = info.external_attr >> 16
        if name.startswith("/") or ".." in parts or stat.S_ISLNK(mode):
            raise ValueError(f"Unsafe artifact member: {info.filename}")
        target = (RELOAD_DIR / Path(name)).resolve()
        if root != target and root not in target.parents:
            raise ValueError("Artifact path escapes destination")
    archive.extractall(RELOAD_DIR)

served = json.loads((RELOAD_DIR / "artifact.json").read_text())
served_ckpt = RELOAD_DIR / served["checkpoint"]
served_context = RELOAD_DIR / served["trainingContext"]
if sha256_file(served_ckpt) != served["digests"]["checkpointSha256"] or sha256_file(served_context) != served["digests"]["trainingContextSha256"]:
    raise RuntimeError("Artifact digest mismatch")

context = pd.read_parquet(served_context)
reloaded = TabICLClassifier(
    model_path=str(served_ckpt), allow_auto_download=False,
    n_estimators=served["inference"]["nEstimators"], random_state=served["inference"]["randomState"],
    device=DEVICE, support_many_classes=True)
reloaded.fit(context[served["featureColumns"]], context[served["targetColumn"]])

smoke_rows = holdout_encoded[FEATURE_COLUMNS].iloc[:min(8, len(holdout_encoded))]
if not np.array_equal(np.asarray(ACTIVE_MODEL.predict(smoke_rows)), np.asarray(reloaded.predict(smoke_rows))):
    raise RuntimeError("Prediction mismatch")
if not np.allclose(np.asarray(ACTIVE_MODEL.predict_proba(smoke_rows)), np.asarray(reloaded.predict_proba(smoke_rows)), rtol=1e-5, atol=1e-7):
    raise RuntimeError("Probability mismatch")
print("✓ Exported bundle reloads and reproduces smoke predictions.")


## What a successful run proves, and what to do next

This session has shown that the checkpoint you evaluated is the pinned release byte for byte, that your data passed the schema, class-coverage and size checks, what the in-context classifier scores on your holdout (and independent test, if you had one), and that the exported bundle reloads from disk and reproduces its predictions. It has **not** shown that the model is fit for your decision; that needs your own held-out data from the population you will deploy on, a cost-aware metric and, for consequential uses, subgroup and drift checks.

### Recap against the objectives
- *Acquire and prove the checkpoint:* Step 2.
- *Prepare data honestly:* Step 3 (preserved splits, schema alignment, training-fitted encoders).
- *Evaluate and read the metrics; selection rules:* Step 4 and the notes after it.
- *Export and prove the bundle:* Steps 6–7.

### Next experiments, in the order they teach the most
1. Upload your own CSV; compare the most-common-class baseline the notebook prints with the pretrained accuracy. That gap is what TabICL gives you for free.
2. Same data as pre-split files with a time- or group-based split; compare with the random split. The difference is the leakage a random split hides.
3. On a table where the pretrained model scores below 0.9, set `RUN_FINE_TUNING = True` on a GPU and see whether a holdout gain survives on the independent test.
4. Feed the exported ZIP to the [artifact-inference notebook](tabiclv2_classifier_artifact_inference_colab.ipynb) with a fresh CSV.

### Troubleshooting
| Symptom | Cause | What to do |
|---|---|---|
| `Checkpoint SHA-256 mismatch` | the file is not the pinned release | re-download, or upload the DIMER release; never edit the expected digest |
| `Expected exactly one .ckpt` | the DIMER ZIP holds several checkpoints or none | upload the base-checkpoint ZIP, not a serving bundle |
| `… contains duplicate column names` | ambiguous CSV header | rename the duplicate columns |
| `holdout schema does not match train` / `target classes unseen in training` | split files disagree | make the feature sets identical; keep every class in training |
| `TabICLv2 fine-tuning requires CUDA` | CPU runtime | *Runtime ▸ Change runtime type ▸ GPU*, or leave `RUN_FINE_TUNING` off |
| fine-tuned metrics identical to pretrained | nothing left to learn on this data; early stopping kept the initial weights | expected on the sample; try a harder table |
| `Artifact digest mismatch` in Step 7 | the ZIP was altered after export | re-run Step 6 |

### Licences and provenance
TabICLv2 and the `tabicl` package are BSD-3-Clause from the Soda team at Inria; the sample is scikit-learn's BSD-licensed Breast Cancer Wisconsin dataset; the exported bundle embeds your training rows and inherits their terms. `artifact.json` records the base checkpoint identity, versions, and authorship.


## AI provenance

This tutorial was developed with substantial AI assistance under maintainer direction and review: original build by **GPT-5.6 Sol High**, via **OpenAI / ChatGPT**, under Agent Relay role **Builder**; content revision (readable code, explanations, troubleshooting) by **Claude Fable 5.1**, via **Anthropic / Claude Code**, under Agent Relay role **Reviewer and Builder**. Attribution is provenance, not sign-off or independent verification.

Upstream model/code: TabICLv2 / `tabicl`, Soda team at Inria, BSD-3-Clause.
